# Advanced vector retrieval strategies

## 1) Imports and functions

In [8]:
import requests
import re
import pdfplumber
import os
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from openai import DefaultHttpxClient, DefaultAsyncHttpxClient
import httpx

load_dotenv()

True

In [2]:
llm_model = ChatOpenAI(
    model=os.getenv("MODEL_LLM"),
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("BASE_URL"),
    streaming=True,
    http_client=DefaultHttpxClient(verify=False),
    http_async_client=DefaultAsyncHttpxClient(verify=False),
)

## 2) Step-back prompting

In [7]:
def generate_stepback(question: str, llm_model: ChatOpenAI, stepback_system_message: str):
    user_message = f"""{question}"""

    messages = [
        ("system", stepback_system_message),
        ("user", user_message)
    ]

    step_back_question = llm_model.invoke(
        messages
    )

    return step_back_question

In [4]:
stepback_system_message = f"""
You are an expert at world knowledge. Your task is to step back and paraphrase a question to a more generic step-back question, which is easier to answer. Here are a few examples:

"input": "Could the members of The Police perform lawful arrests?"
"output": "what can the members of The Police do?"

"input": "Jan Sindel's was born in what country?"
"output": "what is Jan Sindel's personal history?"
"""

In [6]:
question = "Which team did Thierry Audel play for from 2007 to 2008?"

step_back_question = generate_stepback(question, llm_model, stepback_system_message)

print("Question: ", question)
print("Stepback question: ", step_back_question.content)

Question:  Which team did Thierry Audel play for from 2007 to 2008?
Stepback question:  What teams did Thierry Audel play for?


## 3) Parent document retriever

In [10]:
text = ""
pdf_filename = "./data/ch02-downloaded.pdf"

with pdfplumber.open(pdf_filename) as pdf:
    for page in pdf.pages:
        text += page.extract_text()

print(f"Extraction successfully completed. Sample: '{text[:50]}...'")

Extraction successfully completed. Sample: 'Einstein’s Patents and Inventions
Asis Kumar Chaud...'


In [9]:
def split_text_by_titles(text):
    title_pattern = re.compile(r"(\n\d+[A-Z]?\. {1,3}.{0,60}\n)", re.DOTALL)
    titles = title_pattern.findall(text)
    sections = re.split(title_pattern, text)
    sections_with_titles = []
    sections_with_titles.append(sections[0])
    for i in range(1, len(titles) + 1):
        section_text = sections[i*2 - 1].strip() + "\n" + sections[i * 2].strip()
        sections_with_titles.append(section_text)
    return sections_with_titles

In [11]:
sections = split_text_by_titles(text)
print("Number of sections: ", len(sections))

Number of sections:  9
